Objective:

Combine all docs in a directory in a knowledge base and pass relevant information as context to LLM.

The knowledge base is of form:

file name: content

We will do a keyword search in the KB to get relevant context for the LLM.

In [10]:
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [48]:
knowledge_base = {}

for file in Path("./knowledge-base/").rglob('*.md'):
    if 'employees' in file.parent.name or 'products' in file.parent.name:
        file_name = file.name.split(".")[0].split()[-1]
        parent_folder = file.parent.name
        
        with open(file, 'r', encoding='utf-8') as f:
            knowledge_base[file_name.lower()] = f.read()

In [79]:
MODEL = "gpt-4.1-mini"
api_key = load_dotenv("OPENI_API_KEY")
openai = OpenAI()

In [70]:
SYSTEM_PREFIX = """You are an expert on InsureLM, a company that provides insurance related products.
You are able to answer questions related to InsureLM, its products and employees. 
You are provided additional context that might be relevant to a user's question. 
Give brief, accurate answers.
If you don't know the answer to any question, just say so.
Respond in less than 100 words.
"""

In [72]:
def get_additional_context(query):
    context = [knowledge_base[word] for word in query.lower().strip().split(" ") if word in knowledge_base]
    if not context:
        return "There is no additional context relevant to the user's question."

    else:
        return "The following additional context might be relevant to the user's question: \n\n" + \
            "\n\n ".join(context)

In [80]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + get_additional_context(message)
    messages = [{'role':'system', 'content': system_message}] +\
            history +\
                [{'role':'user', 'content':'message'}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [82]:
gr.ChatInterface(
    chat
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
